# DataPulse AI — Final End-to-End Verification

**Purpose:** complete the reproducible synthetic ML demo, validate project tests, optionally verify GCP BigQuery and Pub/Sub, and explicitly distinguish live proof from code-only integrations. Vertex AI and Gemini **are not required**; the DataForge project is deliberately separate. Use your existing trial project. Google Cloud charges may consume eligible trial credits. Do not upgrade the billing account automatically.


## 1 — Get the latest source, install and test

Use this new notebook rather than a stale Colab session. Run from top to bottom; no API key required for the first two sections.


In [ ]:
from pathlib import Path
import subprocess, os
repo=Path('/content/aegis-real-time-supply-chain-ai')
if not (repo/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Codewithadi21/aegis-real-time-supply-chain-ai.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
os.chdir(repo/'datapulse-ai')
print('Root:',os.getcwd())
%pip -q install -r requirements.txt
!python -m pytest -q


## 2 — Generate reliable local ML evidence

Creates three files inside ignored `artifacts/`. The datasets are synthetic. Same model + same holdout cohort are used for before/after comparisons.


In [ ]:
!python -m datapulse --rows 5000
from pathlib import Path
import json
summary=json.loads(Path('artifacts/replay_summary.json').read_text())
report=json.loads(Path('artifacts/incident_report.json').read_text())
print('Schema passed after semantic shift:',summary['schema_checks_after']['passed'])
print('Semantic contract passed after shift:',summary['semantic_after']['passed'])
print('Prediction flips:',summary['prediction_flips'])
print('ROC-AUC before/after:',summary['baseline_model']['roc_auc'],summary['changed_data_same_model']['roc_auc'])
assert summary['schema_checks_after']['passed'] and not summary['semantic_after']['passed']


## 3 — Confirm the Project Guide works without an LLM

This is **source-attributed retrieval only**, not AI-generated text. Qwen LLM inference is separately available in `DataPulse_Guide_AI_Colab.ipynb` and is optional to finish this demo.


In [ ]:
from datapulse.guide_assistant import answer_guide
answer=answer_guide('How does DataPulse detect semantic drift and explain the model impact?',report=report)
print(answer['answer'][:2800])
print('Source documents:',answer['sources'])


## 4 — Optional GCP authentication

Before proceeding: confirm trial balance and project ID in Cloud Billing. Enable BigQuery API and Pub/Sub API in this project. **Project ID is NOT the display name**. Running the next cells may use trial credits. If you don't want cloud operations, stop here; the local demo is complete.


In [ ]:
from google.colab import auth
PROJECT_ID='REPLACE_WITH_YOUR_GCP_PROJECT_ID'
assert PROJECT_ID!='REPLACE_WITH_YOUR_GCP_PROJECT_ID','Enter your real Project ID'
auth.authenticate_user()
%pip -q install -r requirements-gcp.txt
print('Authenticated for:',PROJECT_ID)


## 5 — Write baseline, shifted data and prediction replay to BigQuery

Creates `datapulse_ai.baseline_customers`, `shifted_customers`, `paired_prediction_replay` in US location. Data is synthetic.


In [ ]:
!python -m cloud.bigquery_load --project-id {PROJECT_ID}


## 6 — Verify BigQuery rows and actual decision flips

This query confirms warehouse outputs from your currently authenticated project. Save its output as evidence if it succeeds.


In [ ]:
from google.cloud import bigquery
bq=bigquery.Client(project=PROJECT_ID)
query=f'''SELECT COUNT(*) total_rows, COUNTIF(predicted_churn_before != predicted_churn_after) changed_predictions FROM `{PROJECT_ID}.datapulse_ai.paired_prediction_replay`'''
rows=list(bq.query(query,location='US').result())
assert rows and rows[0].total_rows > 0
print('Verified BigQuery replay rows:',rows[0].total_rows)
print('Verified BigQuery prediction flips:',rows[0].changed_predictions)


## 7 — Create Pub/Sub topic AND subscription BEFORE publishing

This matters: a subscription created after publishing may miss prior events. The demo is one-shot, not an always-on stream.


In [ ]:
!python -m cloud.incident_consumer --project-id {PROJECT_ID} --setup-only


## 8 — Publish one synthetic incident event

The publisher builds a small synthetic replay, sends its metrics and prints a Pub/Sub message ID.


In [ ]:
!python -m cloud.pubsub_demo --project-id {PROJECT_ID}


## 9 — Pull the event, load it into BigQuery, then acknowledge

The consumer validates the Pub/Sub payload, appends to `datapulse_ai.incident_events`, and acknowledges only after successful BigQuery load. If no event is available, publish a new one after creating the subscription and rerun. Duplicate delivery is possible in an at-least-once demo.


In [ ]:
!python -m cloud.incident_consumer --project-id {PROJECT_ID}


## 10 — Verify incident history actually exists


In [ ]:
rows=list(bq.query(f'''SELECT pubsub_message_id, event_type, prediction_flips, semantic_discrepancy_rate, created_at, ingested_at FROM `{PROJECT_ID}.datapulse_ai.incident_events` ORDER BY ingested_at DESC LIMIT 5''',location='US').result())
for row in rows: print(dict(row.items()))
assert len(rows)>0,'No incident rows found in BigQuery'
print('END-TO-END CLOUD VALIDATION PASSED')


## 11 — What to capture for your portfolio

1. Screenshot successful local pytest count and CLI replay metrics.
2. Screenshot the real incident JSON and retrieval source paths.
3. Screenshot the BigQuery query output showing changed predictions.
4. Screenshot Pub/Sub publish message ID and successful acknowledgement.
5. Screenshot the BigQuery incident events table.
6. Optional: run `DataPulse_Guide_AI_Colab.ipynb` for a pretrained Qwen explanation. Not necessary for cloud event flow.

**Not claimed:** a production subscription worker, continuous ingestion, guaranteed exactly-once processing, new foundation LLM training, deployed Vertex AI model, production customer impact. Keep the trial project private and delete cloud resources after the demo if desired.
